# Stage 4 — With GAN, With Drift
### GPVS-Faults | Journal extension of ICSPCS 2024 & ITNAC 2026

Stage 4 = Stage 2's drift injection (`drift_injection.py`, unchanged) +
Stage 3's GAN augmentation (`gan_augmentation.py`, unchanged) + the same two
classifiers. This is the roadmap's **Cell 4 — the main research question**:
does GAN augmentation mitigate drift-induced damage?

**Pipeline ordering matters and is worth stating explicitly.** Drift
injection happens to the real data FIRST (identical to Stage 2), and the
GAN is trained on that already-drifted real training partition SECOND.
This is required, not just convenient: `wgans.py`'s `augment()` zero-fills
the Time column for synthetic rows, so there is no meaningful Time value to
compute a drift tau from at augmentation time. Injecting drift before the
GAN ever sees the data sidesteps that entirely for the baseline variant
(4a) — the GAN just learns and reproduces the already-drifted training
distribution, real Time not required.

## Two variants in this notebook

**Stage 4a — vanilla GAN + drift (the literal 2×2×2 Cell 4).** The GAN
learns the drifted TRAINING distribution (local tau roughly 0–0.625, per
Stage 2's per-class timeline) and generates more of the same. Tests whether
simply having more (synthetic, same-distribution) training data helps the
classifier generalize better to the drifted TEST distribution (tau roughly
0.8125–1.0) — a real question, but the GAN never explicitly sees anything
resembling test-time severity.

**Stage 4b — drift-aware synthetic augmentation ("domain adaptation").**
Takes 4a's synthetic rows and additionally projects them forward along the
same parametric drift ramp (`project_to_tau`), sampling tau from the TEST
partition's range instead of the range the GAN actually learned from. This
gives the classifier labeled synthetic examples resembling the TARGET
(test-time) distribution during training — using the fact that the exact
parametric drift mechanism is known, which is a much stronger assumption
than a generic augmentation method gets to make, but is exactly what this
study's controlled design licenses. **This is the more novel variant and
the one worth featuring as the "domain adaptation" contribution if it beats
4a by more than seed noise.**

**Recommended reporting**: run both. 4a is the necessary baseline cell for
the 2×2×2 table regardless of outcome; 4b is the interesting row if — and
only if — it beats 4a by a margin that survives `multiseed.paired_diff`,
given the ~4.7-pt LSTM-XGB seed noise already established in Stage 2.

> **Execution note:** as in Stage 3, every GPU-bound cell here is ready-to-run
> but not executed in this sandbox. The drift-injection and manipulation-check
> cells (Phase 3/4, unchanged from Stage 2) *were* executed for real, against
> the actual `base_splits.pkl`.


In [1]:
import pandas as pd, numpy as np, os, sys, torch, torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
sys.path.insert(0, "..")
from base_splits import load_splits, summarize_splits
from drift_injection import (detrend_vdc_normal, inject_drift, calibrate_amplitude,
                              manipulation_check, DRIFT_FEATURES,
                              VDC_FLOOR_W_RAW, VDC_FLOOR_KS_RAW)
from gan_augmentation import gan_augment_splits, splits_train_to_array, project_to_tau
from multiseed import run_multi_seed, aggregate_results, summary_table, per_class_accuracy, paired_diff, DEFAULT_SEEDS
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from scipy.stats import ks_2samp, wasserstein_distance
from models.lstm_xgb import LSTM_XGB
from models.cnn_lstm import CNN_LSTM_v1, CNN_LSTM_v2
from utils import reset_gpu_peak_memory, get_gpu_peak_memory_mb, measure_inference_time, Timer, count_parameters

splits_raw = load_splits(path="../base_splits.pkl")
print(summarize_splits(splits_raw))

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

cols = list(splits_raw[0].train.columns[1:-1])
print("feature cols:", cols)


Loaded base splits for 8 classes from /home/maddie/cd-study/Stages/base_splits.pkl
   label  train_n  val_n  test_n  time_start   time_end
0      0     1000    300     300    3.993639   4.153525
1      1     1000    300     300    9.156422   9.316307
2      2     1000    300     300    4.507887   4.667771
3      3     1000    300     300    4.866492   5.026376
4      4     1000    300     300    1.528859   1.688743
5      5     1000    300     300    8.582310   8.742194
6      6     1000    300     300    8.765543   8.925427
7      7     1000    300     300   10.680525  10.840409
device: NVIDIA A30
feature cols: ['Ipv', 'Vpv', 'Vdc', 'ia', 'ib', 'ic', 'va', 'vb', 'vc', 'Iabc', 'If', 'Vabc', 'Vf']


## Phase 3/4 — Drift Injection (unchanged from Stage 2)

Identical code and identical calibrated amplitudes to Stage 2 — the point
of Stage 4 is to add GAN augmentation on top of the *same* drift condition,
not a different one. Re-executed here rather than loaded from a pickle so
this notebook is self-contained.


In [2]:
splits_detrended, vdc_trend_info = detrend_vdc_normal(splits_raw, normal_label=0)
for lbl in range(1, 8):
    assert (splits_detrended[lbl].train["Vdc"].to_numpy()
            == splits_raw[lbl].train["Vdc"].to_numpy()).all()
print("Fault-class Vdc rows confirmed untouched by detrending.")
for tag, s in [("raw", splits_raw), ("detrended", splits_detrended)]:
    tr, te = s[0].train["Vdc"].to_numpy(), s[0].test["Vdc"].to_numpy()
    print(f"Normal Vdc train-vs-test [{tag:>10}]  "
          f"KS={ks_2samp(tr, te).statistic:.4f}  W={wasserstein_distance(tr, te):.4f}")


Fault-class Vdc rows confirmed untouched by detrending.
Normal Vdc train-vs-test [       raw]  KS=0.3807  W=0.1609
Normal Vdc train-vs-test [ detrended]  KS=0.1550  W=0.0938


In [3]:
scaler_ref = StandardScaler().fit(splits_raw[0].train[cols])
sigma = dict(zip(cols, scaler_ref.scale_))

TARGET_SEVERITY_X = 4.0
floor_z = VDC_FLOOR_W_RAW / sigma["Vdc"]
target_z = TARGET_SEVERITY_X * floor_z
print(f"target ({TARGET_SEVERITY_X}x floor, z-units): {target_z:.4f}\n")

amplitudes = {}
for feat in DRIFT_FEATURES:
    target_raw = target_z * sigma[feat]
    amp, w = calibrate_amplitude(splits_detrended, feat, target_raw, scope="all")
    amplitudes[feat] = amp
    print(f"  {feat:5s}  amplitude={amp:.5f}  achieved_mean_W={w:.5f}  (target={target_raw:.5f})")

splits_stage2, tau_lookup = inject_drift(splits_detrended, amplitudes,
                                         features=DRIFT_FEATURES, scope="all")
print("\nDrift injected -- splits_stage2 ready (identical to Stage 2's dataset).")


target (4.0x floor, z-units): 6.9384

  Ipv    amplitude=1.18171  achieved_mean_W=0.69683  (target=0.68754)
  Vpv    amplitude=3.40364  achieved_mean_W=2.02585  (target=2.02635)
  Iabc   amplitude=0.03516  achieved_mean_W=0.02146  (target=0.02163)

Drift injected -- splits_stage2 ready (identical to Stage 2's dataset).


## Stage 4a — GAN Augmentation on the Drifted Training Data

Same `gan_augment_splits` call as Stage 3, fed `splits_stage2` (drifted)
instead of `splits_raw`. The GAN learns whatever the drifted training
partition looks like — it has no knowledge that drift was injected, it
just sees a training distribution and imitates it.


In [4]:
GAN_SEED = 20260827          # same fixed seed as Stage 3, for consistency
GAN_AUGMENT_RATIO = 1.0

splits_stage4a, gans_4, gan_histories_4, n_real = gan_augment_splits(
    splits_stage2, seed=GAN_SEED, device=device, ratio=GAN_AUGMENT_RATIO)

print(f"Trained {len(gans_4)} per-class WGAN-GP generators on drifted training data.")
for lbl, hist in sorted(gan_histories_4.items()):
    print(f"  class {lbl}: final W_dist={hist['w_dist'][-1]:.4f}")


Trained 8 per-class WGAN-GP generators on drifted training data.
  class 0: final W_dist=0.3064
  class 1: final W_dist=0.2970
  class 2: final W_dist=0.2694
  class 3: final W_dist=0.2215
  class 4: final W_dist=0.2787
  class 5: final W_dist=0.3188
  class 6: final W_dist=0.3025
  class 7: final W_dist=0.2736


In [5]:
fidelity_rows = []
for lbl in sorted(splits_stage4a):
    real = splits_stage2[lbl].train
    aug = splits_stage4a[lbl].train
    synth = aug.iloc[len(real):]
    for feat in cols:
        ks = ks_2samp(real[feat], synth[feat]).statistic
        w = wasserstein_distance(real[feat], synth[feat])
        fidelity_rows.append({"class": lbl, "feature": feat, "ks": ks, "wasserstein": w})
fidelity = pd.DataFrame(fidelity_rows)
print("GAN fidelity (drifted real train vs. synthetic), mean across classes:")
print(fidelity.pivot_table(index="feature", values=["ks", "wasserstein"], aggfunc="mean").round(4))


GAN fidelity (drifted real train vs. synthetic), mean across classes:
             ks  wasserstein
feature                     
Iabc     0.0866       0.0005
If       0.0840       0.0099
Ipv      0.0544       0.0167
Vabc     0.0800       0.0055
Vdc      0.2116       0.1556
Vf       0.0814       0.0005
Vpv      0.0621       0.0772
ia       0.0520       0.0610
ib       0.0524       0.0442
ic       0.0585       0.0426
va       0.0474       6.7433
vb       0.0474       6.1659
vc       0.0546       6.8377


## Stage 4b — Drift-Aware Synthetic Augmentation

Take 4a's synthetic rows (which resemble the drifted TRAINING distribution,
local tau roughly 0–0.625) and additionally push them forward to the TEST
partition's tau range (~0.8125–1.0) using the same parametric ramp
(`value - amplitude * tau`) that generated the drift in the first place.
Same real rows, same total synthetic count, same GAN — the only difference
from 4a is where along the drift trajectory the synthetic rows sit.

`TEST_TAU_RANGE` below is the approximate tau span of the test partition
under uniform time sampling (300 of 1600 rows, i.e. the last 18.75%): each
class's test partition spans roughly tau∈[0.8125, 1.0]. This is an
approximation — computable exactly per class from `tau_lookup` above if
tighter precision is wanted.


In [6]:
TEST_TAU_RANGE = (0.8125, 1.0)
rng = np.random.default_rng(GAN_SEED)

splits_stage4b = {lbl: splits_stage4a[lbl] for lbl in splits_stage4a}  # shallow; train replaced below
import copy
splits_stage4b = copy.deepcopy(splits_stage4a)

for lbl in sorted(splits_stage4b):
    aug = splits_stage4b[lbl].train
    n_real_lbl = len(splits_stage2[lbl].train)
    real_part = aug.iloc[:n_real_lbl]
    synth_part = aug.iloc[n_real_lbl:].reset_index(drop=True)
    tau_sample = rng.uniform(TEST_TAU_RANGE[0], TEST_TAU_RANGE[1], size=len(synth_part))
    synth_projected = project_to_tau(synth_part, amplitudes, DRIFT_FEATURES, tau=tau_sample)
    splits_stage4b[lbl].train = pd.concat([real_part, synth_projected], axis=0).reset_index(drop=True)

print("splits_stage4b ready: same real rows + drift-aware-projected synthetic rows.")
print(f"Synthetic rows projected to tau ~ Uniform{TEST_TAU_RANGE} "
      f"(vs. 4a's implicit tau ~ training range).")


splits_stage4b ready: same real rows + drift-aware-projected synthetic rows.
Synthetic rows projected to tau ~ Uniform(0.8125, 1.0) (vs. 4a's implicit tau ~ training range).


## Frozen Scaler + Z-Scaling (both variants)

Same frozen scaler as every stage. Built twice — once for 4a, once for 4b —
from the same unrefit Stage-1 fit.


In [7]:
scaler = StandardScaler()
scaler.fit(splits_raw[0].train[cols])   # frozen -- identical fit to every prior stage

def build_dct(splits_variant):
    dct = dict()
    for i in range(len(splits_variant)):
        dct[i] = dict()
        dct[i].update({
            "train": pd.DataFrame(scaler.transform(splits_variant[i].train[cols]), columns=cols,
                                  index=splits_variant[i].train.index).assign(Fault=i),
            "val": pd.DataFrame(scaler.transform(splits_variant[i].val[cols]), columns=cols,
                                index=splits_variant[i].val.index).assign(Fault=i),
            "test": pd.DataFrame(scaler.transform(splits_variant[i].test[cols]), columns=cols,
                                 index=splits_variant[i].test.index).assign(Fault=i),
        })
    return dct

dct_4a = build_dct(splits_stage4a)
dct_4b = build_dct(splits_stage4b)
print("dct_4a:", {i: len(dct_4a[i]["train"]) for i in dct_4a})
print("dct_4b:", {i: len(dct_4b[i]["train"]) for i in dct_4b})


dct_4a: {0: 2000, 1: 2000, 2: 2000, 3: 2000, 4: 2000, 5: 2000, 6: 2000, 7: 2000}
dct_4b: {0: 2000, 1: 2000, 2: 2000, 3: 2000, 4: 2000, 5: 2000, 6: 2000, 7: 2000}


## Model Evaluation — Both Variants

Unchanged model code. Run once against `dct_4a`, once against `dct_4b`.


In [8]:
def to_tensors(df, cols):
    """Convert a (features + Fault) DataFrame into model-ready tensors.
    X: (N, 1, len(cols)) so the LSTM sees the len(cols) features as a
       length-len(cols) sequence with 1 channel each (matches LSTM_XGB's
       expected input shape).
    y: (N,) integer Fault labels.
    """
    X = torch.from_numpy(df[cols].to_numpy(dtype="float32")).unsqueeze(1)
    y = torch.from_numpy(df["Fault"].to_numpy(dtype="int64"))
    return X, y


def load_scenario(dct, cols):
    """Build combined 8-class train/val/test tensors directly from the
    in-memory `dct` dict (built in the Z-scale cell), instead of reading
    per-scenario CSVs off disk.

    dct is keyed by class label: dct[i]["train"/"val"/"test"] is a
    per-class DataFrame of z-scored features + a Fault column. We
    concatenate across classes to get the full multiclass split.
    """
    train_df = pd.concat([dct[i]["train"] for i in sorted(dct)], axis=0)
    val_df   = pd.concat([dct[i]["val"]   for i in sorted(dct)], axis=0)
    test_df  = pd.concat([dct[i]["test"]  for i in sorted(dct)], axis=0)
    return (to_tensors(train_df, cols),
            to_tensors(val_df,   cols),
            to_tensors(test_df,  cols))


def run_scenario(scenario_idx, dct, cols, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)

    model = LSTM_XGB(n_classes=8, lstm_hidden=32, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)  # LSTM only
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} (LSTM_XGB) ===")
        print(f"  LSTM params: {n_params:,}  breakdown: {params_by_type}")

    # ---- Stage 1: LSTM training ----
    with Timer(device) as stage1_timer:
        model.fit_lstm(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr, weight_decay=weight_decay,
                       batch_size=50, seed=seed, verbose=verbose)

    # ---- Stage 2: XGBoost fitting ----
    with Timer(device=None) as stage2_timer:   # XGBoost is CPU-bound
        model.fit_xgb(X_tr, y_tr)

    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed
    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(LSTM)={stage1_timer.elapsed:.1f}s  "
              f"stage2(XGB)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario": scenario_idx, "model": "LSTM_XGB",
        "n_train": len(X_tr),
        "accuracy": acc, "precision": p, "recall": r, "f1": f,
        "confusion": cm,
        "n_params": n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


### LSTM-XGB — Stage 4a (vanilla GAN + drift)

In [9]:
print(f"LSTM-XGB (4a) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_4a_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct_4a, cols=cols, device=device)
lstm_xgb_4a_agg = aggregate_results(lstm_xgb_4a_results)
print("\nmean +/- std:")
for m, s in lstm_xgb_4a_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


LSTM-XGB (4a) across 5 seeds: [0, 1, 2, 3, 4]
  seed=0  acc=0.7875  P=0.7230  R=0.7875  F1=0.7377  train_sec=90.4
  seed=1  acc=0.7246  P=0.7030  R=0.7246  F1=0.6461  train_sec=90.0
  seed=2  acc=0.6971  P=0.7186  R=0.6971  F1=0.6608  train_sec=89.8
  seed=3  acc=0.7167  P=0.7312  R=0.7167  F1=0.6516  train_sec=89.9
  seed=4  acc=0.7562  P=0.7782  R=0.7562  F1=0.6960  train_sec=90.5

mean +/- std:
  accuracy    0.7364 +/- 0.0356
  precision   0.7308 +/- 0.0284
  recall      0.7364 +/- 0.0356
  f1          0.6784 +/- 0.0384


### LSTM-XGB — Stage 4b (drift-aware / domain adaptation)

In [10]:
print(f"LSTM-XGB (4b) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_4b_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct_4b, cols=cols, device=device)
lstm_xgb_4b_agg = aggregate_results(lstm_xgb_4b_results)
print("\nmean +/- std:")
for m, s in lstm_xgb_4b_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


LSTM-XGB (4b) across 5 seeds: [0, 1, 2, 3, 4]
  seed=0  acc=0.9725  P=0.9767  R=0.9725  F1=0.9723  train_sec=81.8
  seed=1  acc=0.9617  P=0.9705  R=0.9617  F1=0.9609  train_sec=89.0
  seed=2  acc=0.9346  P=0.9476  R=0.9346  F1=0.9347  train_sec=91.5
  seed=3  acc=0.9808  P=0.9831  R=0.9808  F1=0.9810  train_sec=91.4
  seed=4  acc=0.9958  P=0.9959  R=0.9958  F1=0.9958  train_sec=91.9

mean +/- std:
  accuracy    0.9691 +/- 0.0230
  precision   0.9748 +/- 0.0179
  recall      0.9691 +/- 0.0230
  f1          0.9689 +/- 0.0230


In [11]:
# CNN-LSTM training pipeline -- byte-identical to stage1.ipynb. to_tensors()
# and load_scenario() are reused as-is from the previous cell.

def make_model(model_cls, device, seed=0, **kwargs):
    """Fresh CNN-LSTM with Xavier init for Conv/Linear; default PyTorch init
    for LSTM (MATLAB's Glorot applies to Conv and FC, LSTM uses its own)."""
    torch.manual_seed(seed)
    model = model_cls(n_classes=8, **kwargs).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def run_scenario_cnn_lstm(scenario_idx, dct, cols, model_cls, device,
                          epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(model_cls, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({model_cls.__name__}) ===")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":    scenario_idx,
        "model":       model_cls.__name__,
        "n_train":     len(X_tr),
        "accuracy":    acc,
        "precision":   p,
        "recall":      r,
        "f1":          f,
        "confusion":   cm,
        "n_params":    n_params,
        "train_sec":   round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


### CNN-LSTM-v2 — Stage 4a (vanilla GAN + drift)

In [12]:
MODEL_CLS = CNN_LSTM_v2

print(f"CNN-LSTM-v2 (4a) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_4a_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct_4a, cols=cols,
    model_cls=MODEL_CLS, device=device)
cnn_lstm_4a_agg = aggregate_results(cnn_lstm_4a_results)
print("\nmean +/- std:")
for m, s in cnn_lstm_4a_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


CNN-LSTM-v2 (4a) across 5 seeds: [0, 1, 2, 3, 4]
  seed=0  acc=0.8746  P=0.8114  R=0.8746  F1=0.8327  train_sec=140.0
  seed=1  acc=0.8712  P=0.8023  R=0.8712  F1=0.8277  train_sec=128.2
  seed=2  acc=0.8738  P=0.8070  R=0.8738  F1=0.8306  train_sec=122.0
  seed=3  acc=0.8738  P=0.8089  R=0.8738  F1=0.8314  train_sec=124.2
  seed=4  acc=0.8746  P=0.8111  R=0.8746  F1=0.8326  train_sec=124.0

mean +/- std:
  accuracy    0.8736 +/- 0.0014
  precision   0.8081 +/- 0.0037
  recall      0.8736 +/- 0.0014
  f1          0.8310 +/- 0.0021


### CNN-LSTM-v2 — Stage 4b (drift-aware / domain adaptation)

In [13]:
print(f"CNN-LSTM-v2 (4b) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_4b_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct_4b, cols=cols,
    model_cls=MODEL_CLS, device=device)
cnn_lstm_4b_agg = aggregate_results(cnn_lstm_4b_results)
print("\nmean +/- std:")
for m, s in cnn_lstm_4b_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


CNN-LSTM-v2 (4b) across 5 seeds: [0, 1, 2, 3, 4]
  seed=0  acc=0.9542  P=0.9665  R=0.9542  F1=0.9526  train_sec=123.7
  seed=1  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=114.3
  seed=2  acc=0.9279  P=0.9543  R=0.9279  F1=0.9214  train_sec=122.1
  seed=3  acc=0.9979  P=0.9979  R=0.9979  F1=0.9979  train_sec=117.1
  seed=4  acc=0.9992  P=0.9992  R=0.9992  F1=0.9992  train_sec=123.5

mean +/- std:
  accuracy    0.9758 +/- 0.0331
  precision   0.9836 +/- 0.0216
  recall      0.9758 +/- 0.0331
  f1          0.9742 +/- 0.0357


## Combined Summary — All Four Runs

In [14]:
combined = summary_table({
    "LSTM-XGB (4a)": lstm_xgb_4a_agg, "LSTM-XGB (4b)": lstm_xgb_4b_agg,
    "CNN-LSTM-v2 (4a)": cnn_lstm_4a_agg, "CNN-LSTM-v2 (4b)": cnn_lstm_4b_agg,
})
print(combined.round(4).to_string(index=False))

class_names = ["Normal", "F1", "F2", "F3", "F4", "F5", "F6", "F7"]
pc = pd.DataFrame({
    "LSTM-XGB 4a": per_class_accuracy(lstm_xgb_4a_agg["mean_confusion_rate"], class_names),
    "LSTM-XGB 4b": per_class_accuracy(lstm_xgb_4b_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM 4a": per_class_accuracy(cnn_lstm_4a_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM 4b": per_class_accuracy(cnn_lstm_4b_agg["mean_confusion_rate"], class_names),
})
print("\nPer-class accuracy (mean confusion diagonal across seeds):")
print(pc.round(4).to_string())
print("\n>>> Check F3 specifically -- it's the class most worth watching given Stage 2's finding. <<<")

import pickle
with open("stage4_seed_results.pkl", "wb") as f:
    pickle.dump({
        "lstm_xgb_4a": lstm_xgb_4a_agg, "lstm_xgb_4b": lstm_xgb_4b_agg,
        "cnn_lstm_v2_4a": cnn_lstm_4a_agg, "cnn_lstm_v2_4b": cnn_lstm_4b_agg,
    }, f)
print("\nSaved stage4_seed_results.pkl")


           model  n_seeds  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std
   LSTM-XGB (4a)        5         0.7364        0.0356          0.7308         0.0284       0.7364      0.0356   0.6784  0.0384
   LSTM-XGB (4b)        5         0.9691        0.0230          0.9748         0.0179       0.9691      0.0230   0.9689  0.0230
CNN-LSTM-v2 (4a)        5         0.8736        0.0014          0.8081         0.0037       0.8736      0.0014   0.8310  0.0021
CNN-LSTM-v2 (4b)        5         0.9758        0.0331          0.9836         0.0216       0.9758      0.0331   0.9742  0.0357

Per-class accuracy (mean confusion diagonal across seeds):
        LSTM-XGB 4a  LSTM-XGB 4b  CNN-LSTM 4a  CNN-LSTM 4b
Normal       0.2740       0.9693       0.9887       0.9967
F1           1.0000       1.0000       1.0000       1.0000
F2           1.0000       0.9987       1.0000       1.0000
F3           0.0013       0.8287       0.0000       0.8100
F4   

## Key Comparisons (paired by seed)

- **Cell 3 vs. Cell 4a**: does vanilla GAN augmentation mitigate drift damage at all?
- **Cell 2 vs. Cell 4a**: does drift erode the GAN's normal (no-drift) benefit?
- **Cell 4a vs. Cell 4b**: does drift-aware ("domain adaptation") projection
  beat vanilla augmentation, by more than seed noise?


In [17]:
import pickle
with open("../stage2/stage2_seed_results.pkl", "rb") as f:
    stage2 = pickle.load(f)
with open("../stage3/stage3_seed_results.pkl", "rb") as f:
    stage3 = pickle.load(f)

pairs = [
    ("Cell 3 (Stage 2, no GAN) vs Cell 4a (GAN, vanilla)",
     stage2, "lstm_xgb", lstm_xgb_4a_agg, "cnn_lstm_v2", cnn_lstm_4a_agg),
    ("Cell 2 (Stage 3, GAN no-drift) vs Cell 4a (GAN, drift)",
     stage3, "lstm_xgb", lstm_xgb_4a_agg, "cnn_lstm_v2", cnn_lstm_4a_agg),
]
for label, other_stage, key_a, agg_a_lstm, key_b, agg_a_cnn in pairs:
    print(f"\n=== {label} ===")
    for model_key, model_label, agg_new in [
        ("lstm_xgb", "LSTM-XGB", lstm_xgb_4a_agg),
        ("cnn_lstm_v2", "CNN-LSTM-v2", cnn_lstm_4a_agg),
    ]:
        base_results = other_stage[model_key]["raw_results"]
        new_results = agg_new["raw_results"]
        diff_df, diff_stats = paired_diff(new_results, base_results, metric="accuracy")
        print(f"  {model_label}: mean diff = {diff_stats['mean_diff']:+.4f}  "
              f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")

print("\n=== Cell 4a (vanilla) vs Cell 4b (domain-adaptation) ===")
for model_label, results_4a, results_4b in [
    ("LSTM-XGB", lstm_xgb_4a_agg["raw_results"], lstm_xgb_4b_agg["raw_results"]),
    ("CNN-LSTM-v2", cnn_lstm_4a_agg["raw_results"], cnn_lstm_4b_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4b, results_4a, metric="accuracy")
    print(f"  {model_label}: mean diff (4b - 4a) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")



=== Cell 3 (Stage 2, no GAN) vs Cell 4a (GAN, vanilla) ===
  LSTM-XGB: mean diff = -0.0453  t=-1.347  p=0.2494
  CNN-LSTM-v2: mean diff = -0.0005  t=-0.473  p=0.6610

=== Cell 2 (Stage 3, GAN no-drift) vs Cell 4a (GAN, drift) ===
  LSTM-XGB: mean diff = -0.1919  t=-5.730  p=0.0046
  CNN-LSTM-v2: mean diff = -0.1264  t=-206.438  p=0.0000

=== Cell 4a (vanilla) vs Cell 4b (domain-adaptation) ===
  LSTM-XGB: mean diff (4b - 4a) = +0.2327  t=17.964  p=0.0001
  CNN-LSTM-v2: mean diff (4b - 4a) = +0.1022  t=6.811  p=0.0024


## Deferred: Drift-Severity Sub-Matrix (Roadmap §5.2)

The roadmap's full design additionally sweeps TEST severity (no / mild /
matched / severe) specifically for Cell 4, to get a generalization response
curve rather than a single matched-severity point estimate. Not built out
in this notebook (roadmap explicitly deferred "full experimental
parameters... run logistics" pending Phase 4's injection function being
finalized, which it now is) — stubbed here as a concrete next step:

```python
def evaluate_at_severity(dct_builder, splits_variant, severity_multiplier, ...):
    """Re-run inject_drift's amplitude calibration at a different
    TARGET_SEVERITY_X (e.g. 0 for 'no drift', 2.0 for 'mild', 4.0 for
    'matched' -- the value used to train on -- and 6.0 for 'severe'),
    re-scale, and evaluate the ALREADY-TRAINED Stage 4 model against each
    resulting test set. Requires separating 'train severity' (fixed at
    injection time, baked into the trained model) from 'test severity'
    (swept post-hoc against a frozen model) -- i.e. re-inject drift into
    just the TEST partition at each severity level, holding the trained
    model and training data fixed.
    """
```

This is the natural next addition once 4a/4b's matched-severity results are
in and it's clear whether the domain-adaptation variant is worth the extra
complexity of a full severity sweep.


## Next Steps

- Run this notebook end-to-end. The three paired comparisons above are the
  headline results for the paper's Section on Stage 4.
- If 4b's improvement over 4a is within the ~4.7-pt LSTM-XGB seed-noise
  band established in Stage 2, report it as "no significant improvement
  from drift-aware projection" rather than a positive result — the paired
  t-test above is exactly for catching that.
- Track F3 specifically in the per-class table: Stage 2 showed it collapsing
  to ~1-3% regardless of model architecture. Whether Stage 4 recovers it —
  and whether 4a or 4b recovers it better — is likely the most
  reviewer-visible result in this stage.
